# RFT-0008B — R3 Pro4-hint-conditioned Qwen rollout on A100

Current best solver is the Pro4-hint R2 adapter. This notebook uses it to
generate four independent Qwen solutions for hard **official clean-train**
questions, defined by R2-nohint's strict success count of 0–2 out of 4.

The prior Pro4 solution is supplied only as a private generation hint. The
saved R3 corpus contains Qwen outputs only, and accepts a trace only when
its terminal boxed integer equals the official train label. Validation
IDs/templates are excluded. No leaderboard/test/submission data are read.

In [ ]:
# Cell 1 — One-time setup in a fresh A100 runtime.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
%pip uninstall -q -y torchcodec
print("[SETUP] Complete. If vLLM had already been imported, restart once, then run Cell 2 onward.")

In [ ]:
# Cell 2 — Resolve only clean-train, fixed holdouts, prior train rollouts, and Pro4 hints.
import ctypes, difflib, glob, hashlib, json, math, os, re, site, subprocess, sys, time, unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from google.colab import drive

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "main"
RUN_ID = "RFT-0008B-r3-pro4hint-r2solver-k4"
R2_SOLVER_RUN_ID = "RFT-0004B-r2-pro4-hint-lowdrift-lora"
R2_NOHINT_RUN_ID = "RFT-0005A-r2-nohint-laneb-k4-full"
R1_RUN_ID = "RFT-0002-r1-native-k4-full"
SPLIT_RUN_ID = "AUDIT-0002-clean-split-passN-20260821-215706"
SEED = 20260825

N_SAMPLES = 4
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 4096
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 256
MAX_BATCHED_TOKENS = 65536
GPU_MEMORY_UTILIZATION = 0.92
MAX_VISIBLE_WORDS = 350
MAX_SFT_TOKENS = 2048
MAX_PATHS_PER_QUESTION = 2
HARD_MAX_STRICT_SUCCESSES = 2
PROMPT_VERSION = "r2_pro4_private_hint_qwen_rewrite_v1"

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def compact_name(value):
    return re.sub(r"[\s_-]+", "", unicodedata.normalize("NFC", str(value)).casefold())

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(1024 * 1024): digest.update(chunk)
    return digest.hexdigest()

def read_csv(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    frame.columns = [c.strip() for c in frame.columns]
    return frame

def normalize_int(value):
    value = str(value or "").strip().replace(",", "")
    return str(int(value)) if re.fullmatch(r"-?\d+", value) else None

def normalize_question(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value)).casefold()).strip()

def template_key(value):
    return re.sub(r"\d+(?:\.\d+)?", "#", normalize_question(value))

BOX_RE = re.compile(r"\\boxed\s*\{\s*(-?\d(?:[\d,]*\d)?)\s*\}")
def terminal_boxed(value):
    text = str(value or "")
    hits = list(BOX_RE.finditer(text))
    if not hits: return None
    hit = hits[-1]
    tail = text[hit.end():].strip()
    tail = re.sub(r"^(?:\\\)|\\\]|\$\$|\$)+", "", tail).strip()
    tail = re.sub(r"^[.!]+$", "", tail).strip()
    return normalize_int(hit.group(1)) if not tail else None

def load_jsonl_map(path):
    rows = {}
    if not Path(path).exists(): return rows
    with Path(path).open(encoding="utf-8") as handle:
        for number, line in enumerate(handle, 1):
            if not line.strip(): continue
            try: record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(f"[JSONL] ignoring malformed tail {path}:{number}: {exc}"); continue
            if isinstance(record, dict) and record.get("id"): rows[str(record["id"])] = record
    return rows

def append_jsonl(path, rows):
    with Path(path).open("a", encoding="utf-8") as handle:
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush(); os.fsync(handle.fileno())

mount = Path("/content/drive")
if not (mount / "MyDrive").exists(): drive.mount(str(mount))
drive_root = mount / "MyDrive"
projects = [p for p in drive_root.iterdir() if p.is_dir() and compact_name(p.name) == compact_name("2026소중한챌린지")]
assert len(projects) == 1, projects
PROJECT_DIR = projects[0]; RUNS_DIR = PROJECT_DIR / "runs"
data_dirs = [p for p in [PROJECT_DIR / "data", PROJECT_DIR / "Data"] if p.exists()]
clean_paths = [p / "clean_train.csv" for p in data_dirs if (p / "clean_train.csv").exists()]
assert len(clean_paths) == 1, clean_paths
CLEAN_TRAIN_PATH = clean_paths[0]

SPLIT_DIR = RUNS_DIR / SPLIT_RUN_ID / "splits"
TUNE_PATH, DEV_PATH, TEST_HOLDOUT_PATH = [SPLIT_DIR / f"{n}_v1.csv" for n in ["tune", "dev", "test"]]
assert all(p.exists() for p in [TUNE_PATH, DEV_PATH, TEST_HOLDOUT_PATH])

ADAPTER_PATH = RUNS_DIR / R2_SOLVER_RUN_ID / "adapter_final"
assert (ADAPTER_PATH / "adapter_config.json").exists(), ADAPTER_PATH
adapter_config = json.loads((ADAPTER_PATH / "adapter_config.json").read_text(encoding="utf-8"))
adapter_base_model = str(adapter_config.get("base_model_name_or_path", "")).rstrip("/")
assert adapter_base_model == BASE_MODEL or adapter_base_model.endswith("/Qwen2.5-3B-Instruct"), adapter_base_model
adapter_weight = next((ADAPTER_PATH / n for n in ["adapter_model.safetensors", "adapter_model.bin"] if (ADAPTER_PATH / n).exists()), None)
assert adapter_weight is not None

R2_RAW_PATH = RUNS_DIR / R2_NOHINT_RUN_ID / "candidates" / "r2_nohint_laneb_rollouts_shard_00_of_01.jsonl"
R1_VERIFIED_PATH = RUNS_DIR / R1_RUN_ID / "data" / "r1_native_verified_full.csv"
assert R2_RAW_PATH.exists() and R1_VERIFIED_PATH.exists(), (R2_RAW_PATH, R1_VERIFIED_PATH)

specs = []
for p in RUNS_DIR.glob("EXP-0007*/verified/upstage_pro4_verified_cot*.csv"): specs.append((0, p))
for data_dir in data_dirs:
    for p in data_dir.rglob("upstage_pro4_verified_cot*.csv"): specs.append((0, p))
    for p in data_dir.rglob("teacher_core*.csv"): specs.append((1, p))
valid = []
for priority, p in sorted(set(specs), key=lambda x: str(x[1])):
    try: probe = read_csv(p)
    except Exception: continue
    if {"id", "question", "answer", "solution"}.issubset(probe.columns) and len(probe): valid.append((priority, -len(probe), p))
assert valid, "No verified Pro4 train CSV found under project data/runs."
PRO4_PATH = sorted(valid)[0][2]

EXP_DIR = RUNS_DIR / RUN_ID
DATA_DIR, CAND_DIR, REPORT_DIR = [EXP_DIR / x for x in ["data", "candidates", "reports"]]
for p in [DATA_DIR, CAND_DIR, REPORT_DIR]: p.mkdir(parents=True, exist_ok=True)
RAW_PATH = CAND_DIR / "r3_pro4hint_r2solver_rollouts_k4.jsonl"

forbidden = [p for p in [CLEAN_TRAIN_PATH, TUNE_PATH, DEV_PATH, TEST_HOLDOUT_PATH, R2_RAW_PATH, R1_VERIFIED_PATH, PRO4_PATH] if any(x in p.name.casefold() for x in ["leaderboard", "submission"])]
assert not forbidden, forbidden
assert torch.cuda.is_available(), "A100 GPU runtime required."
print("[INPUT] clean:", CLEAN_TRAIN_PATH)
print("[INPUT] R2 nohint rollout:", R2_RAW_PATH)
print("[INPUT] verified Pro4 hints:", PRO4_PATH)
print("[SOLVER] R2 Pro4-hint adapter:", ADAPTER_PATH)
print("[SAFE] leaderboard/test/submission files are not read")

In [ ]:
# Cell 3 — Make R3 hard targets: R2-nohint strict 0–2 successes of 4, with verified Pro4 hints.
clean = read_csv(CLEAN_TRAIN_PATH)
for frame in [clean]:
    assert {"id", "question", "answer"}.issubset(frame.columns) and frame["id"].is_unique
    frame["answer"] = frame["answer"].map(normalize_int); assert frame["answer"].notna().all()
    frame["template_key"] = frame["question"].map(template_key)
holdouts = pd.concat([read_csv(TUNE_PATH), read_csv(DEV_PATH), read_csv(TEST_HOLDOUT_PATH)], ignore_index=True)
holdout_ids = set(holdouts["id"])
holdout_templates = set(holdouts["question"].map(template_key))

prior = load_jsonl_map(R2_RAW_PATH)
assert len(prior) >= 13000, f"Unexpected R2 raw coverage: {len(prior)}"
difficulty = []
for qid, record in prior.items():
    official = normalize_int(record.get("official_answer"))
    candidates = record.get("candidates", [])
    if official is None or len(candidates) != 4: continue
    strict = sum(terminal_boxed(c.get("raw_output", "")) == official for c in candidates)
    difficulty.append({"id": qid, "prior_strict_successes": strict, "prior_rollout_k": 4})
difficulty = pd.DataFrame(difficulty); assert difficulty["id"].is_unique

pro4 = read_csv(PRO4_PATH).drop_duplicates("id", keep="last")
pro4["answer"] = pro4["answer"].map(normalize_int)
pro4["pro4_terminal"] = pro4["solution"].map(terminal_boxed)
pro4["qkey"] = pro4["question"].map(normalize_question)
clean["qkey"] = clean["question"].map(normalize_question)
target_audit = clean.merge(difficulty, on="id", how="inner", validate="one_to_one").merge(
    pro4[["id", "answer", "solution", "pro4_terminal", "qkey"]].rename(columns={"answer":"pro4_answer", "solution":"pro4_hint", "qkey":"pro4_qkey"}),
    on="id", how="left", validate="one_to_one")
target_audit["is_holdout"] = target_audit["id"].isin(holdout_ids) | target_audit["template_key"].isin(holdout_templates)
target_audit["has_verified_hint"] = target_audit["pro4_answer"].eq(target_audit["answer"]) & target_audit["pro4_terminal"].eq(target_audit["answer"]) & target_audit["pro4_qkey"].eq(target_audit["qkey"])
target_audit["is_hard"] = target_audit["prior_strict_successes"].le(HARD_MAX_STRICT_SUCCESSES)
targets = target_audit[target_audit["is_hard"] & ~target_audit["is_holdout"] & target_audit["has_verified_hint"]].copy()
targets = targets.sort_values(["prior_strict_successes", "id"]).reset_index(drop=True)
assert not set(targets.id) & holdout_ids
assert not set(targets.template_key) & holdout_templates
assert targets.pro4_terminal.eq(targets.answer).all()
TARGET_PATH = DATA_DIR / "r3_pro4hint_hard_targets_0to2of4.csv"
AUDIT_PATH = DATA_DIR / "r3_target_eligibility_audit.csv"
targets[["id","question","answer","pro4_hint","prior_strict_successes","prior_rollout_k","template_key"]].to_csv(TARGET_PATH,index=False,encoding="utf-8")
target_audit[["id","prior_strict_successes","prior_rollout_k","is_hard","is_holdout","has_verified_hint"]].to_csv(AUDIT_PATH,index=False,encoding="utf-8")
print("[TARGET] R2 hard (0–2/4) with verified hints:", len(targets))
print(targets.groupby("prior_strict_successes").size())
print("[SAVED]", TARGET_PATH)

In [ ]:
# Cell 4 — Load the current best R2 Pro4-hint LoRA into a fast A100 vLLM engine.
runtime_libs, nvrtc_libs = [], []
for d in site.getsitepackages():
    runtime_libs += glob.glob(str(Path(d) / "nvidia" / "cu13" / "lib" / "libcudart.so.13*"))
    nvrtc_libs += glob.glob(str(Path(d) / "nvidia" / "cu13" / "lib" / "libnvrtc.so.13*"))
runtime_libs, nvrtc_libs = sorted(set(runtime_libs)), sorted(set(nvrtc_libs))
assert runtime_libs and nvrtc_libs, "CUDA13 Python wheels absent. Run Cell 1, restart runtime, then start at Cell 2."
ctypes.CDLL(runtime_libs[0], mode=ctypes.RTLD_GLOBAL); ctypes.CDLL(nvrtc_libs[0], mode=ctypes.RTLD_GLOBAL)
lib_path = ":".join([str(Path(runtime_libs[0]).parent), str(Path(nvrtc_libs[0]).parent), os.environ.get("LD_LIBRARY_PATH", "")])
os.environ["LD_LIBRARY_PATH"] = lib_path; os.environ["LIBRARY_PATH"] = lib_path
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=MODEL_REVISION, use_fast=True, token=False)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
torch.backends.cuda.matmul.allow_tf32 = True
for stream, fd in [(sys.stdout, 1), (sys.stderr, 2)]:
    try: stream.fileno()
    except Exception: stream.fileno = lambda value=fd: value
free_gib = torch.cuda.mem_get_info()[0] / 1024**3
assert free_gib >= 35, f"Only {free_gib:.1f} GiB free. Restart runtime before loading vLLM."
llm = LLM(model=BASE_MODEL, revision=MODEL_REVISION, dtype="bfloat16", tensor_parallel_size=1,
          distributed_executor_backend="uni", gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
          max_model_len=MAX_MODEL_LEN, max_num_seqs=MAX_NUM_SEQS, max_num_batched_tokens=MAX_BATCHED_TOKENS,
          enable_chunked_prefill=True, enable_prefix_caching=True, enable_lora=True,
          max_loras=1, max_lora_rank=32, performance_mode="throughput", trust_remote_code=False)
lora_request = LoRARequest("r2_pro4hint_solver", 1, str(ADAPTER_PATH))
print("[VLLM] ready | free before load:", round(free_gib, 1), "GiB | adapter:", ADAPTER_PATH.name)

In [ ]:
# Cell 5 — R3 Qwen rollout. Exact original R2 private-hint rewrite prompt; resume-safe.
def build_hint_prompt(question, teacher_hint):
    content = (
        f"{str(question).strip()}\n\n"
        "Solve the problem independently. A verified reference derivation is supplied\n"
        "below only as a private mathematical hint. Use it to understand the key idea,\n"
        "but do not mention the hint, the reference, a teacher, or this instruction.\n"
        "Re-derive and verify the solution in your own concise style. End with exactly\n"
        "one final line containing the integer answer as \\boxed{INTEGER}.\n\n"
        "Private reference derivation:\n"
        f"{str(teacher_hint).strip()}"
    )
    return tokenizer.apply_chat_template([{"role":"system","content":"You are a helpful assistant that solves math problems step by step."},{"role":"user","content":content}], tokenize=False, add_generation_prompt=True)

existing = load_jsonl_map(RAW_PATH)
unknown = set(existing) - set(targets.id); assert not unknown, list(unknown)[:3]
pending = targets[~targets.id.isin(existing)].copy()
prompt_batch = max(1, MAX_NUM_SEQS // N_SAMPLES)
print(f"[R3 ROLLOUT] done={len(existing)} pending={len(pending)} prompts/batch={prompt_batch}")
started = time.time()
for start in range(0, len(pending), prompt_batch):
    batch = pending.iloc[start:start + prompt_batch]
    prompts = [build_hint_prompt(x.question, x.pro4_hint) for x in batch.itertuples(index=False)]
    lengths = [len(tokenizer(x, add_special_tokens=False)["input_ids"]) for x in prompts]
    assert max(lengths) + MAX_NEW_TOKENS <= MAX_MODEL_LEN, (max(lengths), MAX_NEW_TOKENS)
    outputs = llm.generate(prompts, SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS, seed=SEED + start), lora_request=lora_request, use_tqdm=False)
    rows = []
    for item, plen, result in zip(batch.itertuples(index=False), lengths, outputs):
        rows.append({"id":str(item.id), "official_answer":str(item.answer), "prior_strict_successes":int(item.prior_strict_successes), "prior_rollout_k":int(item.prior_rollout_k), "prompt_tokens":plen, "prompt_version":PROMPT_VERSION, "pro4_hint_sha256":hashlib.sha256(str(item.pro4_hint).encode()).hexdigest(), "candidates":[{"sample_index":i,"strict_terminal_answer":terminal_boxed(out.text),"generated_tokens":len(out.token_ids),"finish_reason":str(out.finish_reason or ""),"raw_output":out.text} for i,out in enumerate(result.outputs)]})
    append_jsonl(RAW_PATH, rows); existing.update({r["id"]:r for r in rows})
    rate = len(existing) / max(time.time()-started, 1e-9)
    print(f"[R3 ROLLOUT] {len(existing)}/{len(targets)} rate={rate:.2f}q/s eta={(len(targets)-len(existing))/rate/60:.1f}m", flush=True)
assert set(existing) == set(targets.id)
print("[R3 COMPLETE]", RAW_PATH, "sha256=", sha256_file(RAW_PATH))

In [ ]:
# Cell 6 — Strict Qwen-only filtering plus an immutable audit and R3 corpus.
REJECT_PATTERNS = {
    "self_contradiction": re.compile(r"\b(?:contradict|inconsistent|my earlier .*wrong|mistake in)\b", re.I),
    "uncertainty": re.compile(r"\b(?:cannot determine|not enough information|uncertain|unable to solve)\b", re.I),
    "teacher_reference_meta": re.compile(r"\b(?:private reference|teacher|hint provided|as instructed)\b", re.I),
}
def train_prompt(question):
    return tokenizer.apply_chat_template([{"role":"system","content":"You are a helpful assistant that solves math problems step by step."},{"role":"user","content":f"{str(question).strip()}\n\nSolve this step by step, then give the final answer as a single integer inside \\boxed{{}}."}], tokenize=False, add_generation_prompt=True)
def signature(text):
    return re.sub(r"\d+", "#", re.sub(r"\s+", " ", normalize_question(text)))[:2000]

raw = load_jsonl_map(RAW_PATH); lookup = targets.set_index("id").to_dict("index")
audit, accepted, reasons = [], [], Counter()
for qid in targets.id:
    item, record = lookup[qid], raw[qid]; seen = set(); kept = []
    for cand in record["candidates"]:
        text = str(cand["raw_output"]); why = []; terminal = terminal_boxed(text)
        words = len(text.split()); token_count = len(tokenizer(train_prompt(item["question"]) + text, add_special_tokens=False)["input_ids"]); sig = signature(text)
        if terminal != str(item["answer"]): why.append("terminal_boxed_answer_mismatch")
        if words < 20: why.append("solution_too_short")
        if words > MAX_VISIBLE_WORDS: why.append("solution_too_long")
        if token_count > MAX_SFT_TOKENS: why.append("total_tokens_gt_2048")
        if cand.get("finish_reason") == "length": why.append("generation_hit_4096_cap")
        for label, pattern in REJECT_PATTERNS.items():
            if pattern.search(text): why.append(label)
        if sig in seen: why.append("duplicate_reasoning_path")
        for label in why: reasons[label] += 1
        audit.append({"id":qid,"sample_index":cand["sample_index"],"official_answer":item["answer"],"terminal_boxed_answer":terminal or "","word_count":words,"total_tokens":token_count,"decision":"reject" if why else "accept","reasons":" | ".join(why)})
        if not why:
            seen.add(sig); kept.append({"id":qid,"question":item["question"],"answer":str(item["answer"]),"solution":text,"source":"r3_r2_pro4hint_qwen_rewrite","origin":f"r2_nohint_{item['prior_strict_successes']}_of_4","sample_index":cand["sample_index"],"total_tokens":token_count})
    kept.sort(key=lambda x:(x["total_tokens"], x["sample_index"])); accepted.extend(kept[:MAX_PATHS_PER_QUESTION])
audit = pd.DataFrame(audit); verified = pd.DataFrame(accepted)
assert len(verified) and verified.groupby("id").size().max() <= MAX_PATHS_PER_QUESTION
assert all(terminal_boxed(x) == y for x,y in zip(verified.solution,verified.answer))
assert not set(verified.id) & holdout_ids
assert not set(verified.question.map(template_key)) & holdout_templates
CANDIDATE_AUDIT_PATH = DATA_DIR / "r3_pro4hint_r2solver_candidate_audit.csv"
VERIFIED_PATH = DATA_DIR / "r3_pro4hint_r2solver_verified.csv"
audit.to_csv(CANDIDATE_AUDIT_PATH,index=False,encoding="utf-8"); verified.to_csv(VERIFIED_PATH,index=False,encoding="utf-8")
report = {"run_id":RUN_ID,"objective":"R3 hint-conditioned rollout with frozen best R2 Pro4-hint solver on R2-nohint hard train questions (0–2/4).","base_model":BASE_MODEL,"solver_adapter":str(ADAPTER_PATH),"solver_weight_sha256":sha256_file(adapter_weight),"prompt_version":PROMPT_VERSION,"target_questions":len(targets),"verified_questions":int(verified.id.nunique()),"verified_traces":len(verified),"coverage":float(verified.id.nunique()/len(targets)),"reject_reason_counts":dict(reasons),"generation":{"n":N_SAMPLES,"temperature":TEMPERATURE,"top_p":TOP_P,"max_new_tokens":MAX_NEW_TOKENS,"engine":"vllm"},"artifacts":{"targets":str(TARGET_PATH),"raw":str(RAW_PATH),"raw_sha256":sha256_file(RAW_PATH),"candidate_audit":str(CANDIDATE_AUDIT_PATH),"verified":str(VERIFIED_PATH),"verified_sha256":sha256_file(VERIFIED_PATH)},"official_evaluation_files_read":[],"external_api_calls":0,"pro4_usage":"existing official-train verified solution as private rollout hint only; SFT response is Qwen output only."}
REPORT_PATH = REPORT_DIR / "experiment_report.json"; REPORT_PATH.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2)); print("[R3 VERIFIED]", VERIFIED_PATH)

In [ ]:
# Cell 7 — Optional A100 release. Keep False while inspecting/download artifacts.
DISCONNECT_GPU_RUNTIME = True
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True to release the GPU.")